# Evaluation: validation -> settings, then test (issues #6-#8, #32)

1. On the **validation** set: choose the score method, the image threshold and
   the pixel threshold (best MCC), then freeze them.
2. On the **test** set: apply the frozen settings, **once**, and report the
   metrics. If anything is changed after looking at the test results, the test
   is no longer a clean measure.

Results go into the run folder, as `eval_level<dataset level>_trained<training level>/`:
`eval_level1_trained1/` is the run of level 1 on its own level, `eval_level2_trained1/`
the same run on level-2 forgeries, with its own settings kept frozen.

Procedure in `docs/architecture_v0.1.md`. **Runtime: GPU.**

| Cell | What it does |
|---|---|
| 1 | Parameters (your values) |
| 2 | Mount Drive, get the code, install the packages |
| 3 | Copy the dataset to the VM, load the trained model |
| 4 | Validation: choose and save the settings (or, on another level with `REUSE_SETTINGS`, reuse the run's own settings as they are) |
| 5 | Test: metrics with the frozen settings, maps saved |
| 6 | Results per level, scene, shape, border contact, number of pastes (also saved as `metrics_by_<group>.csv`) |
| 7 | Gallery: best successes and most telling failures, each titled with its decision TP / FN / FP / TN (also saved as PNGs in `<EVAL_DIR>/gallery/`) |
| 8 | Predict one patch: a test patch with its true mask by default, or any 512x512 .tif (saved in `<EVAL_DIR>/predictions/`) |

In [ ]:
REPO_URL = "https://github.com/bonlieuguillaume/aletheia.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/aletheia"

TRAINED_ON_LEVEL = 1                               # level the model was trained on (checked in cell 3)
DATASET_LEVEL = 1                                  # level of the forgeries to evaluate on (#24-#27)
# Empty = the default configuration. Any change of the configuration or of
# the code being compared (e.g. "_augment" for the east-west mirror, #37)
# gets a suffix, so that every variant has its own run folder; use the SAME
# suffix as in the unet notebook that trained the run.
RUN_SUFFIX = ""
RUN_NAME = f"v0.1_level{TRAINED_ON_LEVEL}_resnet34{RUN_SUFFIX}"
RUN_DIR = DRIVE_ROOT + "/runs/" + RUN_NAME

# Only used when DATASET_LEVEL != TRAINED_ON_LEVEL:
# True  -> keep the settings chosen on the run's own level (eval_level<T>_trained<T>/),
#          frozen: nothing is tuned on this dataset (pure generalization test);
# False -> "_retuned": the weights stay those learned on TRAINED_ON_LEVEL, but
#          the score method and both thresholds are chosen again on the val of
#          DATASET_LEVEL, then applied to its test. Kept as an option, NOT used
#          in the project: a hybrid that blurs what is measured
#          (docs/architecture_v0.1.md, section 5).
REUSE_SETTINGS = True                              # True throughout the project

DATASET_ZIP = DRIVE_ROOT + f"/data/dataset_v0.1_level{DATASET_LEVEL}.zip"
LOCAL_DATASET = f"/content/dataset_level{DATASET_LEVEL}"   # VM disk, one folder per level
# Inside the run folder: eval_level<dataset level>_trained<training level>.
EVAL_DIR = RUN_DIR + f"/eval_level{DATASET_LEVEL}_trained{TRAINED_ON_LEVEL}"
if DATASET_LEVEL == TRAINED_ON_LEVEL:
    REUSE_SETTINGS = False                         # own level: settings chosen on its val
elif not REUSE_SETTINGS:
    EVAL_DIR += "_retuned"

In [ ]:
# Mount Google Drive, get the latest code from GitHub, install the packages.
# Everything happens on the Colab VM: nothing is installed on your PC.
import os
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/aletheia"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run(["pip", "install", "-q", "-r", REPO_DIR + "/requirements.txt"], check=True)

# The notebook runs from its own folder and imports the module next to it;
# the repository root on sys.path makes `from features.<name>...` work too.
os.chdir(REPO_DIR + "/features/evaluation")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import os

from features.sar_io.sar_io import unzip_folder
from features.unet.unet import load_model

if not os.path.exists(LOCAL_DATASET + "/manifest.csv"):
    unzip_folder(DATASET_ZIP, LOCAL_DATASET)
model, config = load_model(RUN_DIR)
# The level written by the training must match TRAINED_ON_LEVEL (cell 1).
trained_on = config.get("dataset_level", 1)   # runs older than this field: level 1
assert str(trained_on) == str(TRAINED_ON_LEVEL), \
    f"{RUN_NAME} was trained on level {trained_on}: set TRAINED_ON_LEVEL = {trained_on}"
print("Model loaded, commit", config.get("git_commit"))

In [ ]:
import json
from pathlib import Path

from evaluation import choose_thresholds, collect_statistics
from features.unet.unet import iterate_predictions, own_eval_dir

if REUSE_SETTINGS:
    # Generalization test: the settings chosen on the run's own level are
    # applied as they are; this dataset's val is not used.
    own_settings = own_eval_dir(RUN_DIR, config) / "settings.json"
    if not own_settings.exists():
        raise FileNotFoundError(
            f"No settings in {RUN_NAME}/{own_settings.parent.name}/: evaluate this run on "
            f"its own level first (DATASET_LEVEL = {TRAINED_ON_LEVEL}), that is where they are chosen.")
    settings = json.loads(own_settings.read_text())
    settings["reused_from"] = f"{own_settings.parent.name}/settings.json"
    print("Settings reused, frozen:", settings)
else:
    print("Validation set:")
    val_stats = collect_statistics(iterate_predictions(model, config, LOCAL_DATASET, "val"))
    settings = choose_thresholds(val_stats)

Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)
Path(EVAL_DIR + "/settings.json").write_text(json.dumps(settings, indent=2))
config.update(settings)

In [ ]:
from evaluation import compute_metrics, print_summary
from features.sar_io.sar_io import read_manifest

print("Test set (frozen settings):")
test_stats = collect_statistics(iterate_predictions(model, config, LOCAL_DATASET, "test"),
                                save_maps_dir=EVAL_DIR + "/pred_maps")
manifest = read_manifest(LOCAL_DATASET + "/manifest.csv")
summary, results = compute_metrics(test_stats, settings, manifest)
print_summary(summary)

results.to_csv(EVAL_DIR + "/results_test.csv", index=False)
summary_to_save = {k: v for k, v in summary.items()}
# "_ =": write_text returns the number of characters written, which the
# notebook would otherwise print under the summary.
_ = Path(EVAL_DIR + "/summary_test.json").write_text(json.dumps(summary_to_save, indent=2))

In [ ]:
from evaluation import metrics_by_group

# Results per group of forged patches, each also saved as a CSV in EVAL_DIR.
# shape / touches_border exist from level 2 on, n_pastes from level 3 on.
for column in ["level", "scene", "shape", "touches_border", "n_pastes"]:
    if column not in results.columns:
        continue
    table = metrics_by_group(results, column)
    table.to_csv(f"{EVAL_DIR}/metrics_by_{column}.csv")
    print(f"Per {column}:")
    display(table)

In [ ]:
import os

import matplotlib.pyplot as plt

from evaluation import describe_score_method, plot_sample
from features.sar_io.sar_io import read_probability_map, read_tif


# Every figure is also saved as a PNG in <EVAL_DIR>/gallery/ on Drive, so the
# whole gallery can be downloaded at once (one right-click saves one figure).
GALLERY_DIR = EVAL_DIR + "/gallery"
os.makedirs(GALLERY_DIR, exist_ok=True)


# What the image decision made of each patch, written in every title and at
# the end of every file name: a missed forgery (FN) or a false alarm (FP)
# is seen at once, without comparing the score with the threshold by hand.
OUTCOMES = {
    # (true label, predicted label): text
    (1, 1): "TP: detected",
    (1, 0): "FN: MISSED (called authentic)",
    (0, 1): "FP: FALSE ALARM (called forged)",
    (0, 0): "TN: correctly called authentic",
}


def show(row, group):
    patch = read_tif(f"{LOCAL_DATASET}/{row['file']}")
    mask = read_tif(f"{LOCAL_DATASET}/{row['mask_file']}") if row["label"] == 1 else None
    probability_map = read_probability_map(f"{EVAL_DIR}/pred_maps/{row['sample_id']}.tif")
    outcome = OUTCOMES[(int(row["label"]), int(row["predicted_label"]))]
    if row["predicted_label"] == 1:
        comparison = ">="
    else:
        comparison = "<"
    # 3 decimals: the threshold can be small (0.043 at level 2), 2 would hide it.
    title = (f"{row['sample_id']}  {outcome}\n"
             f"score {row['score']:.3f} {comparison} threshold {settings['image_threshold']:.3f} "
             f"({settings['score_method']})")
    if row["label"] == 1:
        title += f"  pixel MCC {row['pixel_mcc']:.2f}"
    if row["predicted_label"] == 0:
        # The map is drawn anyway, but predict() reports no zone for a patch
        # called authentic.
        title += "  (map for information: no zone reported)"
    fig = plot_sample(patch, mask, probability_map, title, config["input_scale"], settings["pixel_threshold"])
    fig.savefig(f"{GALLERY_DIR}/{group}_{row['sample_id']}_{outcome[:2]}.png", dpi=150, bbox_inches="tight")
    plt.show()


# The image score is NOT the mean of all the pixels: it only looks at the
# highest ones (a paste covers 2-5 % of the patch, the rest would pull a mean
# towards 0). Method and threshold chosen on val (cell 4):
print(f"{describe_score_method(settings['score_method'])}; "
      f"'forged' if score >= {settings['image_threshold']:.3f}")

forged = results[results["label"] == 1].sort_values("pixel_mcc")
print("Best forged patches:")
for _, row in forged.tail(3).iterrows():
    show(row, "1_best_forged")
print("Worst forged patches:")
for _, row in forged.head(3).iterrows():
    show(row, "2_worst_forged")
print("Authentic patches with the highest score (false alarms):")
for _, row in results[results["label"] == 0].sort_values("score").tail(3).iterrows():
    show(row, "3_false_alarm_candidates")
print(f"Figures saved in {GALLERY_DIR}")

In [ ]:
# Predict ONE patch. By default: a forged patch of the TEST set (never seen
# in training), shown with its true mask. PATCH_INDEX picks which one
# (0 to 199, in the order of the manifest); `real/` instead of `forged/`
# gives its authentic twin.
# Any 512x512 single-band .tif works too, e.g. the hand-made military-base
# example (#11): set PATCH_PATH to it and MASK_PATH = None.
import os

from features.sar_io.sar_io import read_manifest
from features.unet.unet import predict
from evaluation import describe_score_method

manifest = read_manifest(LOCAL_DATASET + "/manifest.csv")
test_forged = manifest[(manifest["split"] == "test") & (manifest["label"] == 1)]
PATCH_INDEX = 0                                    # 0 .. len(test_forged) - 1
row = test_forged.iloc[PATCH_INDEX]
PATCH_PATH = LOCAL_DATASET + "/" + row["file"]
MASK_PATH = LOCAL_DATASET + "/" + row["mask_file"]

# Uses the settings chosen on val (cell 4); `zone` = pixels called forged.
score, is_forged, probability_map, zone = predict(model, config, PATCH_PATH)
print(describe_score_method(settings["score_method"]))
print(f"{os.path.basename(PATCH_PATH)}: score {score:.3f} -> {'FORGED' if is_forged else 'authentic'} "
      f"(threshold {settings['image_threshold']:.3f}), zone: {zone.mean() * 100:.2f} % of the patch")
mask = read_tif(MASK_PATH) if MASK_PATH else None
decision = "FORGED" if is_forged else "authentic"
title = f"score {score:.3f} -> {decision} (threshold {settings['image_threshold']:.3f})"
fig = plot_sample(read_tif(PATCH_PATH), mask, probability_map, title,
                  config["input_scale"], settings["pixel_threshold"])
# Saved in EVAL_DIR/predictions/, one file per patch looked at (for the slides).
os.makedirs(EVAL_DIR + "/predictions", exist_ok=True)
name = os.path.splitext(os.path.basename(PATCH_PATH))[0]
kind = os.path.basename(os.path.dirname(PATCH_PATH))   # real / forged / your folder
fig.savefig(f"{EVAL_DIR}/predictions/{kind}_{name}.png", dpi=200, bbox_inches="tight")
plt.show()